# F30 Single-Sample Diagnostic

Single-sample notebook for waveform, spectrogram, and harmonic-ridge inspection.


In [46]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
workspace = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(workspace / 'src'))
from f30_fea import AnalysisParams, analyze_sample, load_waveform_record
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.unicode_minus'] = False
%matplotlib qt
import matplotlib as mpl

mpl.rcParams['font.family'] = 'Times New Roman'     # 普通文本
mpl.rcParams['mathtext.fontset'] = 'custom'         # 数学文本用自定义字体
mpl.rcParams['mathtext.rm'] = 'Times New Roman'     # 正常(math roman)
mpl.rcParams['mathtext.it'] = 'Times New Roman:italic'
mpl.rcParams['mathtext.bf'] = 'Times New Roman:bold'

plt.rcParams["font.family"] = "Times New Roman, SimSun" # 显示汉字
plt.rcParams["font.size"] = 16 # 字号


In [47]:
sample_path = workspace / 'data' / 'f130a-pure' / 'F130A-FIP-200K-20260324T100102.344.npz'
output_dir = workspace / 'outputs' / 'f30_single_sample'
output_dir.mkdir(parents=True, exist_ok=True)
params = AnalysisParams(highpass_hz=2_000.0, analysis_band_hz=(5_000.0, 100_000.0), envelope_smooth_ms=1.60, stft_window_ms=0.64, stft_overlap=0.875, 
                        f0_search_hz=(2_000.0, 40_000.0), h2_search_hz=(4_000.0, 80_000.0), h3_search_hz=(12_000.0, 120_000.0), 
                        ridge_jump_penalty_hz=1_000.0, ridge_relative_tolerance=0.08, harmonic_presence_ratio_h2=0.08, harmonic_presence_ratio_h3=0.05, 
                        f0_harmonic_weights=(1.0, 1.35, 0.55))
# 上面参数的含义：
# - highpass_hz: 高频滤波器的截止频率，单位Hz
# - analysis_band_hz: 分析频段的范围，单位Hz
# - envelope_smooth_ms: 包络平滑的时间窗口，单位毫秒
# - stft_window_ms: 短时傅里叶变换的窗口长度，单位毫秒
# - stft_overlap: 短时傅里叶变换的窗口重叠比例，取值范围0-1
# - f0_search_hz: 基频搜索的频率范围，单位Hz
# - h2_search_hz: 二次谐波搜索的频率范围，单位Hz
# - h3_search_hz: 三次谐波搜索的频率范围，单位Hz
# - ridge_jump_penalty_hz: 在频率追踪中，允许的频率跳变的最大值，单位Hz
# - ridge_relative_tolerance: 在频率追踪中，允许的频率跳变相对于当前频率的最大比例
# - harmonic_presence_ratio_h2: 判断二次谐波存在的阈值，单位为二次谐波能量与基频能量的比值
# - harmonic_presence_ratio_h3: 判断三次谐波存在的阈值，单位为三次谐波能量与基频能量的比值
# - f0_harmonic_weights: 在基频追踪中，基频、二次谐波和三次谐波的权重，用于综合考虑它们的能量来确定基频的优先级


In [48]:
record = load_waveform_record(sample_path)
analysis = analyze_sample(record, params=params)
feature_df = pd.DataFrame([analysis.feature_row]).T.reset_index()
feature_df.columns = ['feature', 'value']
feature_df


,feature,value
0,sample_id,F130A-FIP-200K-20260324T100102.344
1,sample_name,F130A-FIP-200K-20260324T100102.344.npz
2,sample_path,E:\codes\ZZ-BK\data\f130a-pure\F130A-FIP-200K-...
3,sample_rate_hz,200000.0
4,signal_duration_ms,11.71
5,raw_ptp,365.397991
6,filtered_rms,0.026869
7,filtered_ptp,0.13694
8,envelope_peak,0.047163
9,event_support_ms,7.49


In [57]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9), constrained_layout=True)
time_ms = analysis.time_s * 1e3
raw_centered = analysis.raw_signal - np.mean(analysis.raw_signal)
raw_scale = np.max(np.abs(raw_centered)) + analysis.params.eps
flt_scale = np.max(np.abs(analysis.filtered_signal)) + analysis.params.eps

# axes[0].plot(time_ms, raw_centered / raw_scale, color='0.75', lw=0.8, label='Raw (normalized)')
axes[0].plot(time_ms, analysis.filtered_signal / flt_scale, color='tab:red', lw=0.9, label='Filtered (normalized)')
axes[0].plot(time_ms, analysis.envelope / (np.max(analysis.envelope) + analysis.params.eps), color='black', lw=1.2, label='Envelope (normalized)')
axes[0].axvline(time_ms[analysis.onset_index], color='tab:green', ls='--', lw=1.0, label='Onset/Offset')
axes[0].axvline(time_ms[analysis.offset_index], color='tab:green', ls='--', lw=1.0)
axes[0].axvline(time_ms[analysis.peak_index], color='tab:blue', ls=':', lw=1.0, label='Envelope peak')
axes[0].set_title(f'Time-domain waveform: {analysis.record.sample_name}', fontsize=20)
axes[0].set_ylabel('Normalized amplitude', fontsize=20)
axes[0].grid(alpha=0.25)
axes[0].legend(loc='upper right', ncol=2)
axes[0].tick_params(labelsize=18)

spectrogram_db = 10.0 * np.log10(analysis.stft_power + analysis.params.eps)
pcm = axes[1].pcolormesh(analysis.stft_times_s * 1e3, analysis.stft_freqs_hz / 1e3, spectrogram_db, shading='auto', cmap='turbo')
axes[1].plot(analysis.stft_times_s * 1e3, analysis.f0_ridge_smooth_hz / 1e3, color='white', lw=1.5, label='f0')
axes[1].plot(analysis.stft_times_s * 1e3, analysis.h2_track.ridge_hz / 1e3, color='cyan', lw=1.1, label='2f')
axes[1].plot(analysis.stft_times_s * 1e3, analysis.h3_track.ridge_hz / 1e3, color='magenta', lw=1.1, label='3f')
if analysis.f0_peak_indices.size:
    axes[1].scatter(analysis.stft_times_s[analysis.f0_peak_indices] * 1e3, analysis.f0_ridge_smooth_hz[analysis.f0_peak_indices] / 1e3, color='yellow', s=28, edgecolors='black', linewidths=0.4, label='f0 peaks')
axes[1].set_title('STFT with detected harmonic ridges', fontsize=20)
axes[1].set_xlabel('Time (ms)', fontsize=20)
axes[1].set_ylabel('Frequency (kHz)', fontsize=20)
axes[1].set_ylim(0, min(100, analysis.params.analysis_band_hz[1] / 1e3))
axes[1].tick_params(labelsize=18)
axes[1].legend(loc='upper right', ncol=2, fontsize=18, edgecolor='black', framealpha=0.3, facecolor='black')
fig.colorbar(pcm, ax=axes[1], label='Power (dB)')
text_features = [f"event_support_ms = {analysis.feature_row['event_support_ms']:.3f}", f"h2_duration_ms = {analysis.feature_row['h2_duration_ms']:.3f}", f"h3_duration_ms = {analysis.feature_row['h3_duration_ms']:.3f}", f"f0_start_khz = {analysis.feature_row['f0_start_khz']:.3f}", f"f0_peak_khz = {analysis.feature_row['f0_peak_khz']:.3f}", f"f0_peak_time_ms = {analysis.feature_row['f0_peak_time_ms']:.3f}", f"f0_end_khz = {analysis.feature_row['f0_end_khz']:.3f}", f"f0_span_khz = {analysis.feature_row['f0_span_khz']:.3f}", f"f0_peak_count = {analysis.feature_row['f0_peak_count']:.0f}", f"f0_arch_score = {analysis.feature_row['f0_arch_score']:.3f}", f"h2_presence_ratio = {analysis.feature_row['h2_presence_ratio']:.3f}", f"h3_presence_ratio = {analysis.feature_row['h3_presence_ratio']:.3f}"]
axes[1].text(1.01, 0.98, '\n'.join(text_features), transform=axes[1].transAxes, va='top', ha='left', fontsize=9, bbox={'boxstyle': 'round', 'facecolor': 'white', 'alpha': 0.85})
fig_path = output_dir / f'{analysis.record.sample_id}_diagnostic.png'
fig.savefig(fig_path, dpi=160, bbox_inches='tight')
print(f'Saved diagnostic figure to: {fig_path}')
plt.show()


Saved diagnostic figure to: E:\codes\ZZ-BK\outputs\f30_single_sample\F130A-FIP-200K-20260324T100102.344_diagnostic.png


# 不同流速时域信号、PSD 对比
